# 🧠 Visual Anomaly Detection — Part 2: Custom Autoencoder from Scratch

**Goal**: Build your OWN anomaly detector using a Convolutional Autoencoder — no Anomalib, pure PyTorch.

**Why this matters**: Understanding HOW anomaly detection works under the hood, not just calling a library.

**Runtime**: `Runtime > Change runtime type > T4 GPU`

---

### How Autoencoder Anomaly Detection Works

```
Normal Image → [Encoder] → Compressed Latent → [Decoder] → Good Reconstruction → Low Error ✅
Anomalous Image → [Encoder] → Compressed Latent → [Decoder] → Bad Reconstruction → High Error ❌
```

The autoencoder only sees normal images during training, so it learns to reconstruct "normal".
When it encounters a defect, the reconstruction is poor → high pixel-wise error = anomaly heatmap.

## 1️⃣ Setup

In [ ]:
!pip install -q anomalib torch torchvision opencv-python matplotlib scikit-learn tqdm

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import cv2
from tqdm.auto import tqdm
import time

from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2️⃣ Load MVTec AD Data

In [ ]:
from anomalib.data import MVTecAD
from torchvision.transforms.v2 import Resize

CATEGORY = "bottle"  # ← Change me!
IMAGE_SIZE = (256, 256)

datamodule = MVTecAD(
    root="./datasets/MVTecAD",
    category=CATEGORY,
    train_batch_size=16,
    eval_batch_size=16,
    num_workers=2,
    augmentations=Resize(IMAGE_SIZE),
)
datamodule.setup()

train_loader = datamodule.train_dataloader()
test_loader = datamodule.test_dataloader()

print(f"✅ Loaded MVTec AD / {CATEGORY}")
print(f"   Train batches: {len(train_loader)}")
print(f"   Test batches:  {len(test_loader)}")

## 3️⃣ Build the Autoencoder Architecture

Architecture:
```
Encoder: [3, 256, 256] → Conv → Pool ×4 → [256, 16, 16] → Flatten → [128] latent
Decoder: [128] → Reshape → [256, 16, 16] → Deconv ×4 → [3, 256, 256]
```

In [ ]:
class ConvBlock(nn.Module):
    """Conv2d → BatchNorm → ReLU."""
    def __init__(self, in_ch, out_ch, kernel_size=3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size, padding=kernel_size//2, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class DeconvBlock(nn.Module):
    """ConvTranspose2d → BatchNorm → ReLU."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class AnomalyAutoencoder(nn.Module):
    """Convolutional Autoencoder for visual anomaly detection.

    Trained on normal images only. Anomalies = high reconstruction error.
    """
    def __init__(self, in_channels=3, latent_dim=128, encoder_channels=None):
        super().__init__()
        if encoder_channels is None:
            encoder_channels = [32, 64, 128, 256]

        self.in_channels = in_channels
        self.latent_dim = latent_dim
        self.encoder_channels = encoder_channels
        decoder_channels = list(reversed(encoder_channels))

        # Encoder: progressively downsample
        encoder_layers = []
        prev_ch = in_channels
        for ch in encoder_channels:
            encoder_layers.append(ConvBlock(prev_ch, ch))
            encoder_layers.append(nn.MaxPool2d(2, 2))
            prev_ch = ch
        self.encoder = nn.Sequential(*encoder_layers)

        # Bottleneck
        self.bottleneck_down = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(encoder_channels[-1], latent_dim),
            nn.ReLU(inplace=True),
        )
        self.bottleneck_up = nn.Sequential(
            nn.Linear(latent_dim, encoder_channels[-1] * 16 * 16),
            nn.ReLU(inplace=True),
        )

        # Decoder: progressively upsample
        decoder_layers = []
        prev_ch = decoder_channels[0]
        for ch in decoder_channels[1:]:
            decoder_layers.append(DeconvBlock(prev_ch, ch))
            prev_ch = ch
        decoder_layers.append(DeconvBlock(prev_ch, prev_ch))
        decoder_layers.append(nn.Conv2d(prev_ch, in_channels, 3, padding=1))
        decoder_layers.append(nn.Sigmoid())
        self.decoder = nn.Sequential(*decoder_layers)

    def encode(self, x):
        features = self.encoder(x)
        return self.bottleneck_down(features)

    def decode(self, z):
        x = self.bottleneck_up(z)
        x = x.view(-1, self.encoder_channels[-1], 16, 16)
        return self.decoder(x)

    def forward(self, x):
        latent = self.encode(x)
        recon = self.decode(latent)
        if recon.shape != x.shape:
            recon = F.interpolate(recon, size=x.shape[2:], mode="bilinear", align_corners=False)
        return recon, latent

    def compute_anomaly_map(self, x):
        """Pixel-wise reconstruction error = anomaly map."""
        self.eval()
        with torch.no_grad():
            recon, _ = self(x)
            error = (x - recon) ** 2
            return error.mean(dim=1, keepdim=True)  # Average across RGB

    def compute_anomaly_score(self, x):
        """Image-level score = max of anomaly map."""
        amap = self.compute_anomaly_map(x)
        return amap.view(x.size(0), -1).max(dim=1)[0]


# Create model
model = AnomalyAutoencoder(in_channels=3, latent_dim=128).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"\n✅ Autoencoder created")
print(f"   Parameters: {total_params:,}")
print(f"   Latent dim: 128")
print(f"   Channels:   [32, 64, 128, 256]")

# Test forward pass
dummy = torch.randn(1, 3, 256, 256).to(device)
recon, latent = model(dummy)
print(f"   Input:  {dummy.shape}")
print(f"   Latent: {latent.shape}")
print(f"   Output: {recon.shape}")

## 4️⃣ Define SSIM Loss

We combine **MSE** (pixel accuracy) + **SSIM** (structural/perceptual quality).

SSIM captures structure that MSE misses — crucial for detecting subtle texture anomalies.

In [ ]:
class SSIMLoss(nn.Module):
    """Structural Similarity Index loss."""
    def __init__(self, window_size=11, channels=3):
        super().__init__()
        self.window_size = window_size
        self.channels = channels
        _1d = self._gaussian(window_size, 1.5).unsqueeze(1)
        _2d = _1d.mm(_1d.t())
        window = _2d.expand(channels, 1, window_size, window_size).contiguous()
        self.register_buffer("window", window)

    @staticmethod
    def _gaussian(size, sigma):
        gauss = torch.tensor([-(x - size//2)**2 / (2*sigma**2) for x in range(size)]).exp()
        return gauss / gauss.sum()

    def forward(self, x, y):
        c1, c2 = 0.01**2, 0.03**2
        pad = self.window_size // 2
        mu_x = F.conv2d(x, self.window, padding=pad, groups=self.channels)
        mu_y = F.conv2d(y, self.window, padding=pad, groups=self.channels)
        sigma_x_sq = F.conv2d(x*x, self.window, padding=pad, groups=self.channels) - mu_x**2
        sigma_y_sq = F.conv2d(y*y, self.window, padding=pad, groups=self.channels) - mu_y**2
        sigma_xy = F.conv2d(x*y, self.window, padding=pad, groups=self.channels) - mu_x*mu_y
        ssim = ((2*mu_x*mu_y + c1) * (2*sigma_xy + c2)) / \
               ((mu_x**2 + mu_y**2 + c1) * (sigma_x_sq + sigma_y_sq + c2))
        return 1 - ssim.mean()


class CombinedLoss(nn.Module):
    """MSE + SSIM combined loss."""
    def __init__(self, ssim_weight=0.5):
        super().__init__()
        self.ssim_weight = ssim_weight
        self.mse = nn.MSELoss()
        self.ssim = SSIMLoss()

    def forward(self, x, y):
        return (1 - self.ssim_weight) * self.mse(x, y) + self.ssim_weight * self.ssim(x, y)


criterion = CombinedLoss(ssim_weight=0.5).to(device)
print("✅ Loss: MSE (50%) + SSIM (50%)")

## 5️⃣ Train the Autoencoder

Key training details:
- **One-class learning**: Training data = normal images ONLY
- **CosineAnnealing LR**: Smooth learning rate decay
- **Early stopping**: Stop when validation loss stops improving

In [ ]:
# Training config
EPOCHS = 100
LR = 0.001
PATIENCE = 15

optimizer = Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

# Training loop
history = {"train_loss": [], "val_loss": [], "lr": []}
best_loss = float("inf")
patience_counter = 0

start_time = time.time()
print(f"🚀 Training autoencoder for {EPOCHS} epochs...\n")

for epoch in range(1, EPOCHS + 1):
    # ---- Train ----
    model.train()
    epoch_loss = 0
    n_batches = 0
    for batch in train_loader:
        images = batch["image"].to(device)
        optimizer.zero_grad()
        recon, _ = model(images)
        loss = criterion(recon, images)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1

    avg_train = epoch_loss / max(n_batches, 1)
    scheduler.step()

    # ---- Validate (on test normal images) ----
    model.eval()
    val_loss = 0
    val_batches = 0
    with torch.no_grad():
        for batch in test_loader:
            images = batch["image"].to(device)
            labels = batch["label"]
            normal_mask = labels == 0
            if normal_mask.sum() == 0:
                continue
            normal_imgs = images[normal_mask]
            recon, _ = model(normal_imgs)
            loss = criterion(recon, normal_imgs)
            val_loss += loss.item()
            val_batches += 1

    avg_val = val_loss / max(val_batches, 1)
    current_lr = scheduler.get_last_lr()[0]

    history["train_loss"].append(avg_train)
    history["val_loss"].append(avg_val)
    history["lr"].append(current_lr)

    if epoch % 10 == 0 or epoch == 1:
        print(f"  Epoch {epoch:3d}/{EPOCHS} │ Train: {avg_train:.6f} │ Val: {avg_val:.6f} │ LR: {current_lr:.2e}")

    # Early stopping
    if avg_val < best_loss:
        best_loss = avg_val
        patience_counter = 0
        best_state = model.state_dict().copy()
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n  ⚡ Early stopping at epoch {epoch}")
            break

elapsed = time.time() - start_time
print(f"\n✅ Training done in {elapsed:.0f}s ({elapsed/60:.1f} min)")

# Load best model
model.load_state_dict(best_state)

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history["train_loss"]) + 1)
ax1.plot(epochs_range, history["train_loss"], label="Train", color="#2196F3", lw=2)
ax1.plot(epochs_range, history["val_loss"], label="Val", color="#FF5722", lw=2)
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("Training & Validation Loss", fontweight="bold")
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs_range, history["lr"], color="#4CAF50", lw=2)
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Learning Rate")
ax2.set_title("Cosine Annealing LR Schedule", fontweight="bold")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6️⃣ Visualize Reconstructions: Normal vs Anomalous

This is the core insight: the autoencoder reconstructs normal images well, but fails on anomalies!

In [ ]:
model.eval()

# Get some test samples
test_images, test_labels, test_recons, test_amaps = [], [], [], []
with torch.no_grad():
    for batch in test_loader:
        images = batch["image"].to(device)
        labels = batch["label"]
        recon, _ = model(images)
        amaps = model.compute_anomaly_map(images)

        test_images.extend(images.cpu())
        test_labels.extend(labels.cpu().numpy().tolist())
        test_recons.extend(recon.cpu())
        test_amaps.extend(amaps.cpu())

# Pick 4 normal + 4 anomalous
normal_idx = [i for i, l in enumerate(test_labels) if l == 0]
anom_idx = [i for i, l in enumerate(test_labels) if l == 1]
rng = np.random.default_rng(42)
picks = list(rng.choice(normal_idx, min(4, len(normal_idx)), replace=False)) + \
        list(rng.choice(anom_idx, min(4, len(anom_idx)), replace=False))

fig, axes = plt.subplots(3, len(picks), figsize=(3 * len(picks), 9))
fig.suptitle(f"Autoencoder Reconstructions — {CATEGORY.title()}", fontsize=15, fontweight="bold")

for col, idx in enumerate(picks):
    img = test_images[idx].permute(1, 2, 0).numpy()
    rec = test_recons[idx].permute(1, 2, 0).numpy()
    amap = test_amaps[idx].squeeze().numpy()
    label = test_labels[idx]

    img = np.clip(img, 0, 1)
    rec = np.clip(rec, 0, 1)

    color = "red" if label == 1 else "green"
    label_str = "❌ Anomalous" if label == 1 else "✅ Normal"

    axes[0, col].imshow(img)
    axes[0, col].set_title(label_str, color=color, fontweight="bold", fontsize=9)
    axes[0, col].axis("off")

    axes[1, col].imshow(rec)
    axes[1, col].set_title("Reconstruction", fontsize=9)
    axes[1, col].axis("off")

    # Heatmap overlay
    if amap.max() > amap.min():
        amap_norm = (amap - amap.min()) / (amap.max() - amap.min())
    else:
        amap_norm = np.zeros_like(amap)
    heatmap = cv2.applyColorMap((amap_norm * 255).astype(np.uint8), cv2.COLORMAP_JET)
    heatmap_rgb = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted((img * 255).astype(np.uint8), 0.6, heatmap_rgb, 0.4, 0)
    axes[2, col].imshow(overlay)
    axes[2, col].set_title("Error Heatmap", fontsize=9)
    axes[2, col].axis("off")

axes[0, 0].set_ylabel("Input", fontsize=12, fontweight="bold")
axes[1, 0].set_ylabel("Recon", fontsize=12, fontweight="bold")
axes[2, 0].set_ylabel("Heatmap", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 7️⃣ Determine Anomaly Threshold & Evaluate

We fit a Gaussian to normal reconstruction errors, then set threshold at **μ + 3σ**.

In [ ]:
# Compute scores on training data (all normal) to find threshold
train_scores = []
model.eval()
with torch.no_grad():
    for batch in train_loader:
        images = batch["image"].to(device)
        scores = model.compute_anomaly_score(images)
        train_scores.extend(scores.cpu().numpy().tolist())

train_scores = np.array(train_scores)
threshold = train_scores.mean() + 3.0 * train_scores.std()
print(f"Normal score distribution: μ={train_scores.mean():.6f}, σ={train_scores.std():.6f}")
print(f"Threshold (μ + 3σ): {threshold:.6f}")

In [ ]:
# Evaluate on test set
all_scores, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        images = batch["image"].to(device)
        labels = batch["label"]
        scores = model.compute_anomaly_score(images)
        all_scores.extend(scores.cpu().numpy().tolist())
        all_labels.extend(labels.numpy().tolist() if hasattr(labels, 'numpy') else labels)

all_scores = np.array(all_scores)
all_labels = np.array(all_labels).astype(int)
predictions = (all_scores > threshold).astype(int)

# Metrics
auroc = roc_auc_score(all_labels, all_scores)
f1 = f1_score(all_labels, predictions, zero_division=0)
precision = precision_score(all_labels, predictions, zero_division=0)
recall = recall_score(all_labels, predictions, zero_division=0)

print(f"\n🏆 Custom Autoencoder Results — {CATEGORY.title()}")
print(f"{'='*40}")
print(f"  Image AUROC:  {auroc:.4f}")
print(f"  F1 Score:     {f1:.4f}")
print(f"  Precision:    {precision:.4f}")
print(f"  Recall:       {recall:.4f}")

In [ ]:
# Score distribution plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
normal_scores = all_scores[all_labels == 0]
anom_scores = all_scores[all_labels == 1]
ax1.hist(normal_scores, bins=30, alpha=0.7, label="Normal", color="#4CAF50", edgecolor="white")
ax1.hist(anom_scores, bins=30, alpha=0.7, label="Anomalous", color="#FF5252", edgecolor="white")
ax1.axvline(threshold, color="black", linestyle="--", lw=2, label=f"Threshold={threshold:.4f}")
ax1.set_xlabel("Anomaly Score")
ax1.set_ylabel("Count")
ax1.set_title("Score Distribution", fontweight="bold")
ax1.legend()

# ROC curve
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(all_labels, all_scores)
ax2.plot(fpr, tpr, color="#2196F3", lw=2, label=f"AUROC = {auroc:.4f}")
ax2.plot([0,1], [0,1], "k--", alpha=0.3)
ax2.set_xlabel("False Positive Rate")
ax2.set_ylabel("True Positive Rate")
ax2.set_title("ROC Curve", fontweight="bold")
ax2.legend(loc="lower right")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8️⃣ Visualize Encoder Feature Maps

Let's peek inside the encoder to see what features it's learning.

In [ ]:
# Hook to capture intermediate features
features_at_layers = {}

def hook_fn(name):
    def hook(module, input, output):
        features_at_layers[name] = output.detach()
    return hook

# Register hooks on each ConvBlock in encoder
hooks = []
layer_idx = 0
for i, layer in enumerate(model.encoder):
    if isinstance(layer, ConvBlock):
        h = layer.register_forward_hook(hook_fn(f"encoder_block_{layer_idx}"))
        hooks.append(h)
        layer_idx += 1

# Run one image through
sample = datamodule.test_data[anomalous_pick[0] if 'anomalous_pick' in dir() else 0]
img_tensor = sample["image"].unsqueeze(0).to(device)
model.eval()
with torch.no_grad():
    _ = model(img_tensor)

# Visualize
n_layers = len(features_at_layers)
fig, axes = plt.subplots(1, n_layers + 1, figsize=(4 * (n_layers + 1), 4))

# Original image
img_np = img_tensor.squeeze().permute(1, 2, 0).cpu().numpy()
axes[0].imshow(np.clip(img_np, 0, 1))
axes[0].set_title("Input", fontweight="bold")
axes[0].axis("off")

# Feature maps (show channel 0)
for i, (name, feat) in enumerate(features_at_layers.items()):
    fmap = feat[0, 0].cpu().numpy()  # First sample, first channel
    axes[i+1].imshow(fmap, cmap="viridis")
    axes[i+1].set_title(f"Block {i}\n{feat.shape[1]}ch, {feat.shape[2]}x{feat.shape[3]}", fontsize=9)
    axes[i+1].axis("off")

fig.suptitle("Encoder Feature Maps (Channel 0)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Remove hooks
for h in hooks:
    h.remove()

## 🎓 What You Just Built

| Component | What It Does |
|-----------|-------------|
| **ConvBlock** | Conv2d → BatchNorm → ReLU building block |
| **Encoder** | 4 conv blocks with max pooling (256×256 → 16×16) |
| **Bottleneck** | AdaptivePool → Linear (compress to 128-dim latent) |
| **Decoder** | 4 deconv blocks (16×16 → 256×256) |
| **SSIM Loss** | Perceptual loss for structural quality |
| **Threshold** | Gaussian fit: μ + 3σ on normal scores |

### Interview Answers
| Question | Your Answer |
|----------|------------|
| Why train on normal only? | Autoencoder learns the "normal" manifold; deviations = anomalies |
| Why SSIM + MSE? | MSE = pixel accuracy, SSIM = structural similarity. Combined catches more |
| How do you set the threshold? | Fit Gaussian to normal errors, threshold = μ + kσ |
| Why is the reconstruction bad for anomalies? | Bottleneck forces compression; model never saw anomalies, can't represent them |

**Next**: `03_benchmark_and_compare.ipynb` — Compare all models side-by-side!